In [ ]:
# get all SSHOMP items with non-english language description


In [3]:
%pip install requests

  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.2-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (35 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.2-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (148 kB)
Using cached idna-3.10-py3-none-any.whl (70 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]4/5 [requests]
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install langdetect


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=1041ec81058935470f076f4970488eac8a00f93d920a6a58222c0126fd0a1ec0
  Stored in directory: /home/michael/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [8]:
#use the new repo to download latest api snapshots
import os
import sys
import json
import requests
import time
import re
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Get the list of .json files from the GitHub repo and download the latest one
repo_api_url = "https://api.github.com/repos/SSHOC/sshompitor/contents/data"
response = requests.get(repo_api_url)
files = response.json()

# Filter for .json files with 'full_items_' in the name and sort by timestamp in the name
json_files = [f for f in files if f['name'].startswith('full_items_') and f['name'].endswith('.json')]
json_files.sort(key=lambda x: int(re.search(r'full_items_(\d+)\.json', x['name']).group(1)))

latest_json = json_files[-1]
download_url = latest_json['download_url']

# Download the latest .json file
latest_json_content = requests.get(download_url).text
print("downloading" + latest_json['name'])

# Save to local file
with open(latest_json['name'], 'w', encoding='utf-8') as f:
    f.write(latest_json_content)

print(f"Downloaded {latest_json['name']}")

downloadingfull_items_1753071443.json
Downloaded full_items_1753071443.json


In [5]:
df = pd.read_json(latest_json['name'])

In [6]:
print(df)

         id         category  \
0     72078  tool-or-service   
1     74226  tool-or-service   
2     72079  tool-or-service   
3     75428  tool-or-service   
4     36324  tool-or-service   
...     ...              ...   
5769  12589          dataset   
5770  76522          dataset   
5771  76239          dataset   
5772  75795          dataset   
5773  75825          dataset   

                                                  label persistentId  \
0                                                140kit       SIU1nO   
1             1641 Depositions - Trinity College Dublin       i60Tk3   
2                  360 stopni: 360-degree documentation       tZUwaR   
3     3D fotogrametria: 3D models of any objects (ph...       tLdtav   
4     3DF Zephyr - photogrammetry software - 3d mode...       4gDAHv   
...                                                 ...          ...   
5769  "You Are Where You Tweet: A Content-Based Appr...       YnEaU0   
5770                    Zurich English 

In [9]:
# in label, find all items with non-english language description using nlp


# Ensure consistent results
DetectorFactory.seed = 0


# Language detection function
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Apply function to each row
df['detected_language'] = df['description'].apply(detect_language)

# Display results
print(df)


         id         category  \
0     72078  tool-or-service   
1     74226  tool-or-service   
2     72079  tool-or-service   
3     75428  tool-or-service   
4     36324  tool-or-service   
...     ...              ...   
5769  12589          dataset   
5770  76522          dataset   
5771  76239          dataset   
5772  75795          dataset   
5773  75825          dataset   

                                                  label persistentId  \
0                                                140kit       SIU1nO   
1             1641 Depositions - Trinity College Dublin       i60Tk3   
2                  360 stopni: 360-degree documentation       tZUwaR   
3     3D fotogrametria: 3D models of any objects (ph...       tLdtav   
4     3DF Zephyr - photogrammetry software - 3d mode...       4gDAHv   
...                                                 ...          ...   
5769  "You Are Where You Tweet: A Content-Based Appr...       YnEaU0   
5770                    Zurich English 